# 01 — `Table`, `Series`, and the shapes a query can take

Everything in this notebook is the **44-name v0 consumer surface**, plus the two
public types (`Series`, `Table`) and the disclosure functions. Every name used
below is listed in `showcase/README.md` with the test that pins it. Nothing here
reaches into `numfast._*` — that is internal by the package's own contract.

The block layout is described **in prose and from `app.schema()` / `explain()`,
which are public**. No private API is introduced to make a point about layout.

In [1]:
import numpy as np
import pandas as pd
import numfast as nf

print("numfast", nf.__version__)
print("native:", nf.native_info())

numfast 0.2.1


native: {'disabled': False, 'dll': 'C:\\App\\numfast\\numfast\\src\\numfast\\_native\\numfast_native.dll', 'dll_exists': True}


## `Series` — one named buffer plus an optional validity sidecar

`from_numpy` on 1-D gives a `Series`; on 2-D it gives a `Table`. That is the only
distinction between the two calls.

In [2]:
s = nf.from_numpy(np.array([3, 1, 4, 1, 5], dtype=np.int32), name="v")
print("name  ", s.name)
print("dtype ", s.dtype)
print("backend", s.backend)
print("values", s.to_numpy().tolist())

name   v
dtype  int32
backend None
values [3, 1, 4, 1, 5]


## `Table` — a dictionary of named series

`nf.Table(kernel, {name: series})`. The **key you write is authoritative**:
every `from_numpy` series defaults to the name `"v"`, and a Table built naively
from them would lose the names. That is pinned by
`test_group_text_key_keeps_the_table_key_not_the_series_name`.

In [3]:
units = nf.from_numpy(np.array([10, 3, 7, 1], dtype=np.int32))
price = nf.from_numpy(np.array([2.5, 9.0, 4.0, 11.0]))
t = nf.Table(nf.get_kernel(), {"units": units, "price": price})

print("names", t.names, "ncols", t.ncols)
print(t.to_pandas().to_string(index=False))

names ['units', 'price'] ncols 2
 units  price
    10    2.5
     3    9.0
     7    4.0
     1   11.0


## The layout the architecture actually supports

`app.schema()` is the honest view: each column has a **logical** type (what the
user means) and a **physical** type (what is stored), plus a `scale`/`offset`
pair and an `encoding`. This is the only layout information the public surface
exposes, and it is enough to see the two facts that matter:

1. a TEXT column is **physically `int32`** — a dictionary *code* vector;
2. an integer column can carry a `scale`/`offset`, i.e. the stored representation
   is allowed to differ from the logical one.

`explain()` shows the second fact at execution level: the IR is a list of jobs,
one node per column, with explicit edges.

In [4]:
app = nf.app()
for name, col in app.schema(t).items():
    print(f"{name:6s} logical={col['logical']:8s} physical={col['physical']:8s}"
          f" scale={col['scale']} offset={col['offset']}"
          f" encoding={col['encoding']['type']}")

units  logical=int32    physical=int32    scale=1 offset=0 encoding=auto
price  logical=float64  physical=float64  scale=1 offset=0 encoding=auto


In [5]:
# A text column is physically int32: the DAG holds dictionary CODES.
plan = pd.DataFrame({"plan": ["a", "b", "a", "c"], "rev": [1.0, 2.0, 3.0, 4.0]})
tp = nf.from_pandas(plan)
for name, col in app.schema(tp).items():
    print(f"{name:6s} logical={col['logical']:8s} physical={col['physical']}")

plan   logical=text     physical=int32
rev    logical=float64  physical=float64


## Arithmetic between columns

`app.c(name)` builds an expression over a column. The operators are the method
names: `+ - * / % **` and `== != < <= > >=`, plus `and_` (`&`), `not_` (`~`).
This is `Expr`, and it is a lazy handle — nothing runs until `compile()`.

In [6]:
ch = t.query().filter(app.c("units") > 2).derive("gross", app.c("units") * app.c("price"))
print(ch.jobs())

[{'op': 'series', 'inputs': [], 'params': {'values': array([10,  3,  7,  1], dtype=int32), 'dtype': 'int32'}, 'out': 's_units_1'}, {'op': 'compare', 'inputs': ['s_units_1'], 'params': {'op': '>', 'value': 2}, 'out': 'cmp_2'}, {'op': 'filter', 'inputs': ['s_units_1', 'cmp_2'], 'params': {}, 'out': 'f_units_3'}, {'op': 'series', 'inputs': [], 'params': {'values': array([ 2.5,  9. ,  4. , 11. ]), 'dtype': 'float64'}, 'out': 's_price_5'}, {'op': 'filter', 'inputs': ['s_price_5', 'cmp_2'], 'params': {}, 'out': 'f_price_4'}, {'op': 'map', 'inputs': ['f_units_3', 'f_price_4'], 'params': {'fn': 'mul'}, 'out': 'm_6'}]


`jobs()` is the semantic IR made visible, and it is the same IR that compiles —
there is no second IR anywhere. Three nodes for three columns: two sources and
one `map`.

In [7]:
print(ch.explain())

EXPLAIN backend=n/a requested=n/a
GRAPH nodes=6 outputs=['m_6'] ops=[series:s_units_1, compare:cmp_2, filter:f_units_3, series:s_price_5, filter:f_price_4, map:m_6]
STATE placement=n/a state=n/a selectivity=n/a dispatches=n/a chunks=n/a
PROFILE version=n/a source=n/a age=n/a matched=n/a warning=none
ESTIMATED cpu=n/ams gpu_host=n/ams chosen=n/a
GPU eligible=n/a blockers=n/a coverage=n/a
EDGE s_units_1 <- [] [series]
EDGE cmp_2 <- ['s_units_1'] [compare]
EDGE f_units_3 <- ['s_units_1', 'cmp_2'] [filter]
EDGE s_price_5 <- [] [series]
EDGE f_price_4 <- ['s_price_5', 'cmp_2'] [filter]
EDGE m_6 <- ['f_units_3', 'f_price_4'] [map]


In [8]:
out = ch.compile()
print(out.to_pandas().to_string(index=False))

 units  price  gross
    10    2.5     25
     3    9.0     27
     7    4.0     28


## Derived columns

`derive` **adds** a column; it never replaces one, and it refuses a name that is
already taken. Expressions nest, so a derived column can feed the next one.

In [9]:
ch2 = t.query().derive("gross", app.c("units") * app.c("price"))
print(ch2.compile().to_pandas().to_string(index=False))

 units  price  gross
    10    2.5     25
     3    9.0     27
     7    4.0     28
     1   11.0     11


Deriving twice is where the surface refuses, and the refusal is precise:
**`truediv` over a derived expression is rejected**, because the DAG has no
way to know a derived expression's dtype, and `ir_map('div')` rounds the
quotient back whenever the left operand's buffer is an integer dtype. v0 has no
dtype-promotion verb — no cast, no `where`, and a float literal does not promote
either. The engine will not promise a float result it cannot deliver. This holds
even when both source columns are float: the limitation is the *derived*
operand, not the dtype.

In [10]:
try:
    t.query().derive("gross", app.c("units") * app.c("price")) \
            .derive("ratio", app.c("gross") / app.c("units")) \
            .compile()
except ValueError as err:
    print("ValueError:", err)

ValueError: derive: truediv over 'gross': the left operand has no declared dtype -- it is a derived expression -- and ir_map('div') rounds the quotient back whenever the LEFT operand's buffer is an integer dtype (Drivers/CPU/_lib/cpu.py:3109-3111), so v0 cannot promise the float result it would be asked for. v0 has no dtype-promotion verb: no cast and no where, and a float literal does not promote either (ir_map casts add/sub/mul back to the left dtype, Drivers/CPU/_lib/cpu.py:3133), so an int column cannot become a float column inside the DAG. Load the column as float (from_pandas / from_numpy with float values), or express the ratio with mul/add. Fix: divide a column that is float at the source: q.c('rate') / q.c('other').


In [11]:
# The suggested spelling: divide two columns that are float at the source.
tf_units = nf.from_numpy(np.array([10.0, 3.0, 7.0, 1.0]))
tf_price = nf.from_numpy(np.array([2.5, 9.0, 4.0, 11.0]))
tf = nf.Table(nf.get_kernel(), {"units": tf_units, "price": tf_price})
divided = (tf.query()
           .derive("unit_price", app.c("price") / app.c("units"))
           .compile())
print(divided.to_pandas().to_string(index=False))

 units  price  unit_price
  10.0    2.5    0.250000
   3.0    9.0    3.000000
   7.0    4.0    0.571429
   1.0   11.0   11.000000


### A deliberate note on dtypes

On the integer table above, `units` is `int32` and `price` is `float64`, and
their product came back with the fractional part **truncated** rather than
refused. That is a real defect in this tree, not a demonstration; it is recorded
under «Находки» in `showcase/README.md` and deliberately used to make no point
at all. The float64 table gives the exact result, bit for bit equal to
pandas.

In [12]:
exact = (tf.query().derive("gross", app.c("units") * app.c("price"))
        .compile())

frame = pd.DataFrame({"units": [10.0, 3.0, 7.0, 1.0], "price": [2.5, 9.0, 4.0, 11.0]})
print("numfast:", exact.column("gross").to_numpy().tolist())
print("numpy  :", (frame.units * frame.price).tolist())
print("equal  :", np.array_equal(exact.column("gross").to_numpy(),
                                 (frame.units * frame.price).to_numpy()))

numfast: [25.0, 27.0, 28.0, 11.0]
numpy  : [25.0, 27.0, 28.0, 11.0]
equal  : True


## Operations across tables

There is **no `join` in the v0 surface** — it is one of four names deliberately
absent, and the README says why: the join primitives exist in the engine but
have no consumer-facing spelling. So an across-tables operation is done with
`nf.lookup`, which is a **kernel-level** name: below the facade, not part of the
44, and it says so when you read it.

`lookup` builds a hash table from `build` and probes it with `probe`. Keys must
be `int32`. It returns `positions` (the index of the match, `-1` for a miss) and
`hit` (a boolean mask).

In [13]:
orders = pd.DataFrame({
    "region":  ["emea", "apac", "emea", "amer"],
    "units":   [10.0, 3.0, 7.0, 1.0],
    "price":   [2.5, 9.0, 4.0, 11.0],
})
regions = pd.DataFrame({"region": ["emea", "apac", "amer"],
                        "label":  ["Europe", "Asia-Pacific", "Americas"]})

codes = np.array([0, 1, 2], dtype=np.int32)
to_code = nf.from_numpy(codes, name="code")

# build = the small side (codes), probe = the big side
lk = nf.lookup(to_code, to_code)
print("keys of the lookup result:", sorted(lk))

keys of the lookup result: ['hit', 'k', 'positions']


In [14]:
# The join itself, done with derive + lookup on int32 codes.
codes_by_region = {"emea": 0, "apac": 1, "amer": 2}
build_side = nf.from_numpy(
    np.array([codes_by_region[r] for r in regions.region], dtype=np.int32),
    name="code")
probe_side = nf.from_numpy(
    np.array([codes_by_region[r] for r in orders.region], dtype=np.int32),
    name="code")

hit = nf.lookup(build_side, probe_side)
print("positions:", hit["positions"].to_numpy().tolist())
print("hit      :", hit["hit"].to_numpy().tolist())

labels = regions.label.to_numpy()
joined = (pd.DataFrame({
    "region": orders.region,
    "units":  orders.units,
    "price":  orders.price,
    "label":  labels[hit["positions"].to_numpy()],
}))
print(joined.to_string(index=False))

positions: [0, 1, 0, 2]
hit      : [True, True, True, True]
region  units  price        label
  emea   10.0    2.5       Europe
  apac    3.0    9.0 Asia-Pacific
  emea    7.0    4.0       Europe
  amer    1.0   11.0     Americas


## Group and reduce

`group(keys, aggs)` takes a tuple of aggregations per measure column. The
measure column name gets the aggregation appended, so `"rev"` with `("sum",
"count")` produces columns `rev.sum` and `rev.count`.

`sort(*keys, desc=False)` and `limit(n, offset=0)` follow. `reduce(op, col)` is
the un-grouped form and returns a **Python scalar**, not a Table.

In [15]:
g = (t.query()
     .group("units", {"price": ("sum", "count", "max")})
     .sort("price.sum", desc=True)
     .compile())
print(g.names)
print(g.to_pandas().to_string(index=False))

['units', 'price.sum', 'price.count', 'price.max']
 units  price.sum  price.count  price.max
     1       11.0            1       11.0
     3        9.0            1        9.0
     7        4.0            1        4.0
    10        2.5            1        2.5


In [16]:
print("reduce sum   :", t.query().reduce("sum", "price"))
print("reduce mean  :", t.query().reduce("mean", "price"))
print("nrows (lazy) :", t.query().nrows())

reduce sum   : 26.5
reduce mean  : 6.625
nrows (lazy) : 4


## The refusal that makes this a library

`group` on a key column containing NULL **raises**, because a NULL group would
be *absent* rather than wrong, and an absent group cannot be told apart from an
answer. `is_null()` exists so the advice is executable — note that `is_null()`
is a NULL **test** (hard True/False on every row) and `not_` is the correct
three-valued negation of a predicate. They share no code path.

In [17]:
nullable = nf.from_pandas(pd.DataFrame({
    "plan": pd.array(["a", None, "b", "a"], dtype="string"),
    "rev":  [1.0, 2.0, 3.0, 4.0],
}))

try:
    nullable.query().group("plan", {"rev": ("sum",)}).compile()
except ValueError as err:
    print("ValueError:", err)

ValueError: group: key column 'plan' has 1 NULL rows; v0 does not group NULL keys -- NULL keys are silently dropped (Drivers/CPU/_lib/cpu.py:3325), so a NULL group is an ABSENT value, not a wrong one, and cannot be told apart from the answer. Fix: filter those rows out before group().


In [18]:
clean = (nullable.query()
         .filter(app.c("plan").is_null().not_())
         .group("plan", {"rev": ("sum", "count")})
         .compile())
print(clean.to_pandas().to_string(index=False))

plan  rev.sum  rev.count
   a      5.0          2
   b      3.0          1

## Kernel-level names on a single `Series`

Below the facade — **not** in the 44-name registry, and they may change shape
between releases. They are listed here because they are real, and because the
distinction is the point.

In [19]:
s = nf.from_numpy(np.array([3, 1, 4, 1, 5], dtype=np.int32), name="v")
print("cumsum :", nf.to_numpy(nf.cumsum(s)).tolist())
print("shift  :", nf.to_numpy(nf.shift(s, 1)).tolist())
print("reduce :", s.reduce("sum"))

u = nf.unique(s)          # a dict, not a Series
print("unique :", u["ng"], list(nf.to_numpy(u["uniq"])))

cumsum : [3, 4, 8, 9, 14]
shift  : [0, 3, 1, 4, 1]
reduce : 14
unique : 4 [np.int32(1), np.int32(3), np.int32(4), np.int32(5)]


## Round trip

Every conversion the surface offers, on one table.

In [20]:
print("to_pandas:"); print(g.to_pandas().to_string(index=False))
print("to_numpy shapes:", {n: g.column(n).to_numpy().shape for n in g.names})
print("schema of result:", sorted(app.schema(g)))

to_pandas:
 units  price.sum  price.count  price.max
     1       11.0            1       11.0
     3        9.0            1        9.0
     7        4.0            1        4.0
    10        2.5            1        2.5
to_numpy shapes: {'units': (4,), 'price.sum': (4,), 'price.count': (4,), 'price.max': (4,)}
schema of result: ['price.count', 'price.max', 'price.sum', 'units']
